# Day 2: Chunk, embed, store, and visualize a knowledge base

## Introduction
Day 1 retrieved documents by exact keywords. Today we prepare for **semantic
retrieval** by converting document chunks into numerical embedding vectors.
Similar meanings can have similar vectors even when the words differ.

**Pipeline:** local Markdown -> LangChain Documents + metadata -> overlapping
chunks -> local MiniLM embeddings -> persistent Chroma collection -> retrieval
smoke test -> 2D and 3D t-SNE plots.

This notebook prepares and inspects a retrieval index. It does **not** call a chat
model or implement answer generation.

### Running this reference
- Select the project's Python environment and run cells top to bottom.
- Start the kernel inside this repository. This notebook deliberately reads your
  copy of `my-work/Week 5/knowledge-base`, not the original course documents.
- Hugging Face model/tokenizer and tiktoken files may download on first use.
  Once cached, embedding runs locally; no OpenAI key or paid API calls are needed.
- Text and vectors are persisted on disk. Use course examples, not confidential
  information. Plot hover text also exposes excerpts from the source documents.
- Existing vector data is not deleted. A dedicated collection is reused only
  when its chunk IDs match; otherwise choose a new collection name explicitly.
- Old outputs have been cleared. Restart the kernel and rerun after editing
  model, chunking, paths, or collection configuration.

## 1. Imports and tokenizer-process setup
Set `TOKENIZERS_PARALLELISM` **before** tokenizer-related imports to avoid the
fork-after-threading warning. This disables tokenizer worker threads, not the
embedding model itself. Restart the kernel if tokenizers were already used.

Import NumPy directly: `wandb.util` is not NumPy's public import path.

In [2]:
import os

# Apply before Hugging Face/tokenizer imports, rather than after initialization.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import hashlib
import html
from pathlib import Path

import numpy as np
import plotly.graph_objects as go
import tiktoken
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
from transformers import AutoTokenizer

## 2. Resolve project paths and choose the embedding model
Paths are anchored to this project's Week 5 folder, not whichever directory the
kernel happens to use. Missing project/corpus paths stop with clear errors.

`sentence-transformers/all-MiniLM-L6-v2` produces 384-dimensional vectors locally.
Its SentenceTransformer configuration limits inputs to 256 tokens. We use a
200-content-token chunk target with 40-token overlap, leaving room for special
tokens below that limit. This intentionally replaces the original 1,000-character
split, because characters do not reliably bound model input tokens.

The collection name describes this configuration. Changing the embedding model
or chunking requires a new collection name (or a deliberate rebuild outside this
reference); vectors from different embedding models must not be mixed.

In [3]:
current_dir = Path.cwd().resolve()
for directory in (current_dir, *current_dir.parents):
    project_dir = directory / "my-work" / "Week 5"
    if (project_dir / "2-day2.ipynb").is_file():
        break
else:
    raise FileNotFoundError(
        f"Could not locate my-work/Week 5/2-day2.ipynb from {current_dir}. "
        "Start the kernel inside the repository."
    )

knowledge_base_dir = project_dir / "knowledge-base"
if not knowledge_base_dir.is_dir():
    raise FileNotFoundError(f"Knowledge-base directory is missing: {knowledge_base_dir}")

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 200
CHUNK_OVERLAP = 40
db_name = str(project_dir / "vector_db")
collection_name = "insurellm_day2_minilm_tokens200_overlap40_v1"
print(f"Reading documents from: {knowledge_base_dir}")
print(f"Chroma directory: {db_name}; collection: {collection_name}")

Reading documents from: /Users/michaelperrine/Udemy/Donner/llm_engineering/my-work/Week 5/knowledge-base
Chroma directory: /Users/michaelperrine/Udemy/Donner/llm_engineering/my-work/Week 5/vector_db; collection: insurellm_day2_minilm_tokens200_overlap40_v1


## Part A: Inspect and chunk documents

### 3. Count files, characters, and approximate tokens
Read sorted Markdown paths for predictable inspection. Joining once is simpler
than repeatedly extending a large string. The concatenation is for counting
only; each source document remains separate during chunking.

`o200k_base` is an OpenAI-style encoding for a **rough corpus-size comparison**.
It is not MiniLM's tokenizer and does not prove a specific chat model's token
usage, context fit, price, or chunk length.

In [4]:
files = sorted(knowledge_base_dir.rglob("*.md"))
if not files:
    raise FileNotFoundError(f"No Markdown files found in {knowledge_base_dir}")

entire_knowledge_base = "\n\n".join(
    file_path.read_text(encoding="utf-8") for file_path in files
)
print(f"Found {len(files)} Markdown files.")
print(f"Total characters, including separators: {len(entire_knowledge_base):,}")

Found 76 Markdown files.
Total characters, including separators: 304,432


In [5]:
# This encoding is for illustration; MiniLM token counts are measured below.
encoding = tiktoken.get_encoding("o200k_base")
token_count = len(encoding.encode(entire_knowledge_base, disallowed_special=()))
print(f"Approximate corpus tokens using o200k_base: {token_count:,}")

Approximate corpus tokens using o200k_base: 63,555


### 4. Load LangChain Documents and preserve provenance
A `Document` holds `page_content` plus `metadata`. `TextLoader` treats Markdown as
UTF-8 text; it does not render Markdown. `DirectoryLoader` reads each category
recursively. Each chunk will inherit its file's `source` and category `doc_type`.

All Markdown files must live under a category folder such as `employees`,
`products`, `company`, or `contracts`. The coverage check detects accidentally
omitted root-level or hidden files instead of silently indexing only part of
the corpus. A new category can be loaded but must get a plot color later.

In [6]:
folders = sorted(folder for folder in knowledge_base_dir.iterdir() if folder.is_dir())
documents = []
for folder in folders:
    loader = DirectoryLoader(
        str(folder),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    for doc in loader.load():
        doc.metadata["doc_type"] = folder.name
        documents.append(doc)

documents.sort(key=lambda doc: doc.metadata["source"])
loaded_sources = {Path(doc.metadata["source"]).resolve() for doc in documents}
if len(documents) != len(files) or loaded_sources != {file.resolve() for file in files}:
    raise ValueError("Not all Markdown files were loaded; check category folders and hidden files.")
if any(not doc.page_content.strip() for doc in documents):
    raise ValueError("The knowledge base contains an empty document; inspect source files.")
print(f"Loaded {len(documents)} documents with source and category metadata.")

Loaded 76 documents with source and category metadata.


In [7]:
# Preview one document without assuming a second document exists.
documents[0]

Document(metadata={'source': '/Users/michaelperrine/Udemy/Donner/llm_engineering/my-work/Week 5/knowledge-base/company/about.md', 'doc_type': 'company'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates wit

### 5. Split using the embedding model's tokenizer
The recursive splitter prefers paragraph, line, and word boundaries, then falls
back to smaller pieces. Its size units here are **MiniLM tokens**, not characters.
The splitter counts content tokens; the budget check also counts the tokenizer's
special tokens (202 total at most for this configuration).
Overlap retains nearby context across boundaries, but 40 is a target rather than
a guarantee of exactly 40 shared tokens for every adjacent pair.

Each chunk retains its source path and category. The explicit length check catches
oversized chunks before embedding can silently truncate them. This is still a
teaching split, not a document-structure-aware parser. Character offsets are not
requested: the installed splitter uses character-based overlap when locating
offsets, which does not reliably match this token-based overlap.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)
chunks = text_splitter.split_documents(documents)
if not chunks:
    raise ValueError("Document splitting produced no chunks.")

chunk_token_counts = [
    len(tokenizer.encode(chunk.page_content, add_special_tokens=True))
    for chunk in chunks
]
token_budget = CHUNK_SIZE + tokenizer.num_special_tokens_to_add(pair=False)
if token_budget > 256 or max(chunk_token_counts) > token_budget:
    raise ValueError("A chunk exceeds the token budget; revise splitting before embedding.")

print(f"Split {len(documents)} documents into {len(chunks)} chunks.")
print(f"Largest chunk: {max(chunk_token_counts)} MiniLM tokens including special tokens (budget {token_budget}).")

Split 76 documents into 449 chunks.
Largest chunk: 202 MiniLM tokens including special tokens (budget 202).


In [9]:
# The original chunks[100] failed for small corpora; a first chunk always exists here.
chunks[0]

Document(metadata={'source': '/Users/michaelperrine/Udemy/Donner/llm_engineering/my-work/Week 5/knowledge-base/company/about.md', 'doc_type': 'company'}, page_content='# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.')

**Part A summary:** the corpus is loaded once per file with provenance attached.
Chunks are overlapping excerpts sized using the actual embedding tokenizer.
Corpus token estimates and embedding-token budgets answer different questions.

## Part B: Embed and persist in Chroma

### 6. Create or reuse a non-destructive collection
Embeddings map each chunk to a dense vector. Normalization gives each vector unit
length, so Euclidean distance on these normalized vectors ranks pairs consistently
with cosine similarity. Chroma persists vectors, text, and metadata automatically.

The original notebook deleted its collection on every run. Here a separate named
collection leaves the old default `langchain` collection untouched. Stable IDs
encode the source path, chunk order, content, and model. An identical rerun
reuses the existing collection without adding duplicates; changed input raises
an error instead of mixing stale and new chunks or deleting data.

This strict snapshot policy is intentional: it is not incremental index updating.
If files, chunking, or embedding settings change, choose a new `collection_name`.
Collection names also distinguish settings not encoded in IDs, such as normalization.

In [10]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

# Relative sources make IDs stable when the project is moved to another machine.
chunk_ids = []
for chunk_index, chunk in enumerate(chunks):
    relative_source = Path(chunk.metadata["source"]).relative_to(knowledge_base_dir).as_posix()
    identity = (
        f"{EMBEDDING_MODEL}\n{relative_source}\n"
        f"{chunk_index}\n{chunk.page_content}"
    )
    chunk_ids.append(hashlib.sha256(identity.encode("utf-8")).hexdigest())
if len(set(chunk_ids)) != len(chunk_ids):
    raise ValueError("Duplicate chunk IDs found; inspect chunk sources and ordering.")

vectorstore = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=db_name,
)
existing_ids = set(vectorstore.get(include=[])["ids"])
if existing_ids and existing_ids != set(chunk_ids):
    raise ValueError(
        "This collection contains a different corpus snapshot. "
        "Choose a new collection_name and rerun; existing data has not been deleted."
    )
if not existing_ids:
    vectorstore.add_documents(documents=chunks, ids=chunk_ids)

print(f"Collection '{collection_name}' contains {len(vectorstore.get(include=[])['ids'])} chunks.")

Collection 'insurellm_day2_minilm_tokens200_overlap40_v1' contains 449 chunks.


### 6b. Fix: embedding-dimension mismatch (384 vs 1536)

The cell above switched `embeddings` to OpenAI `text-embedding-3-small`
(**1536** dimensions), but the existing collection stores **384**-dimensional
MiniLM vectors. A Chroma collection's dimension is fixed when it is created, so
every query must use the *same* embedding model that built it — hence
`InvalidArgumentError: Collection expecting embedding with dimension of 384, got 1536`.

The fix below follows this notebook's own rule — *new embedding model ⇒ new
collection* — and does two things:

1. **Restores `vectorstore`** to the MiniLM collection with MiniLM embeddings,
   so every later cell (dimension check, smoke test, t-SNE, regression asserts)
   works again.
2. **Builds a separate OpenAI collection** (`openai_vectorstore`) with its own
   IDs, so both models can be compared side by side. Embedding the 449 chunks is
   a one-time API call (~90k tokens of `text-embedding-3-small` — well under a
   cent); unchanged re-runs reuse the snapshot, just like the MiniLM collection.


In [ ]:
# ── Fix the dimension mismatch: one collection per embedding model ────────────
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings as OpenAIEmbeddingsV2  # current, non-deprecated package

load_dotenv(override=True)  # this notebook never loaded .env; OpenAI calls need the API key

# 1) Restore `vectorstore` to the MiniLM collection with MATCHING embeddings,
#    so all later cells (384-dim checks, smoke test, t-SNE, asserts) work again.
minilm_embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)
vectorstore = Chroma(
    collection_name=collection_name,
    embedding_function=minilm_embeddings,
    persist_directory=db_name,
)

# 2) Put the OpenAI embeddings in their OWN collection (new model => new name,
#    new IDs — the identity hash includes the model name so snapshots never mix).
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"   # 1536 dimensions
openai_collection_name = "insurellm_day2_openai3small_tokens200_overlap40_v1"
openai_embeddings = OpenAIEmbeddingsV2(model=OPENAI_EMBEDDING_MODEL)

openai_chunk_ids = []
for chunk_index, chunk in enumerate(chunks):
    relative_source = Path(chunk.metadata["source"]).relative_to(knowledge_base_dir).as_posix()
    identity = (
        f"{OPENAI_EMBEDDING_MODEL}\n{relative_source}\n"
        f"{chunk_index}\n{chunk.page_content}"
    )
    openai_chunk_ids.append(hashlib.sha256(identity.encode("utf-8")).hexdigest())

openai_vectorstore = Chroma(
    collection_name=openai_collection_name,
    embedding_function=openai_embeddings,
    persist_directory=db_name,
)
openai_existing = set(openai_vectorstore.get(include=[])["ids"])
if openai_existing and openai_existing != set(openai_chunk_ids):
    raise ValueError(
        "The OpenAI collection contains a different corpus snapshot. "
        "Choose a new collection_name and rerun; existing data has not been deleted."
    )
if not openai_existing:
    openai_vectorstore.add_documents(documents=chunks, ids=openai_chunk_ids)  # one-time embedding API call

print(f"MiniLM collection '{collection_name}': "
      f"{len(vectorstore.get(include=[])['ids'])} chunks (384-dim)")
print(f"OpenAI collection '{openai_collection_name}': "
      f"{len(openai_vectorstore.get(include=[])['ids'])} chunks (1536-dim)")

# 3) Side-by-side smoke test — each store embeds the query with its own model
for label, store in (("MiniLM", vectorstore), ("OpenAI", openai_vectorstore)):
    print(f"\n=== {label} retrieval: 'Who is Avery Lancaster?' ===")
    for match in store.similarity_search("Who is Avery Lancaster?", k=3):
        print(f"{match.metadata['doc_type']} | {match.metadata['source']}")
        print(match.page_content[:150], "\n")

### 7. Inspect stored vectors
Use the public `vectorstore.get()` API instead of relying on the private
`_collection` attribute. Request only one embedding for the dimension check;
the plotting section fetches the complete small teaching collection.

In [11]:
count = len(vectorstore.get(include=[])["ids"])
sample = vectorstore.get(limit=1, include=["embeddings"])
sample_vectors = sample["embeddings"]
if sample_vectors is None or len(sample_vectors) != 1:
    raise ValueError("Expected one stored embedding; check the collection contents.")
dimensions = len(sample_vectors[0])
if dimensions != 384:
    raise ValueError(f"Expected 384 MiniLM dimensions, found {dimensions}.")
print(f"There are {count:,} vectors, each with {dimensions:,} dimensions.")

There are 449 vectors, each with 384 dimensions.


### 8. Retrieval smoke test
Search embeds the question with the same model, then retrieves nearby chunks.
Inspect source paths and text to judge relevance; a returned result is not proof
that it answers the question. No generation model is involved.

In [12]:
# This makes a local embedding/query, not an OpenAI chat request.
matches = vectorstore.similarity_search("Who is Avery Lancaster?", k=min(3, count))
for match in matches:
    print(f"{match.metadata['doc_type']} | {match.metadata['source']}")
    print(match.page_content[:200], "\n")

InvalidArgumentError: Collection expecting embedding with dimension of 384, got 1536

**Part B summary:** Chroma stores one row per chunk, not one row per original
file. An unchanged rerun reuses its snapshot. The dedicated collection and
explicit mismatch error protect existing data, but do not perform migrations.

## Part C: Visualize the stored embeddings

### 9. Prepare aligned vectors, categories, and hover text
Chroma returns embeddings, documents, and metadata aligned with the returned IDs,
not necessarily the input file order. Keep those arrays together. Validate their
shapes, finite values, and categories before fitting a projection.

Hover excerpts are HTML-escaped so Markdown containing `<...>` is displayed as
text. Colors are categorical labels from folder metadata, not predicted classes.

In [ ]:
result = vectorstore.get(include=["embeddings", "documents", "metadatas"])
vectors = np.asarray(result["embeddings"], dtype=float)
chunk_texts = result["documents"]
metadatas = result["metadatas"]
if chunk_texts is None or metadatas is None:
    raise ValueError("Stored text or metadata is missing.")
if vectors.shape != (count, dimensions) or not np.isfinite(vectors).all():
    raise ValueError("Stored embeddings have an unexpected shape or non-finite values.")
if len(chunk_texts) != count or len(metadatas) != count:
    raise ValueError("Text, metadata, and embeddings are not aligned.")

doc_types = [metadata["doc_type"] for metadata in metadatas]
COLOR_MAP = {"products": "blue", "employees": "green", "contracts": "red", "company": "orange"}
unknown_types = set(doc_types) - COLOR_MAP.keys()
if unknown_types:
    raise ValueError(f"Add colors for these document categories: {sorted(unknown_types)}")
colors = [COLOR_MAP[doc_type] for doc_type in doc_types]
hover_text = [
    f"Type: {html.escape(doc_type)}<br>"
    f"Source: {html.escape(Path(metadata['source']).name)}<br>"
    f"Text: {html.escape(text[:100])}..."
    for doc_type, metadata, text in zip(doc_types, metadatas, chunk_texts)
]

### 10. Project into 2D and 3D using t-SNE
t-SNE is a nonlinear visualization, not the retrieval algorithm. It emphasizes
local neighborhoods; axis values, global distances, and apparent cluster sizes
have no direct business meaning. Separate 2D/3D fits are not interchangeable
coordinate systems, and clusters alone do not measure retrieval quality.

Perplexity must be strictly smaller than the number of samples. The helper uses
at most 30 and adapts to small corpora; fewer than two vectors raises an explicit
error. Random initialization also permits 3D output when there are fewer than
three samples/features. `random_state=42` improves repeatability within the same
environment, not numerical identity across every library version.

In [ ]:
def project_vectors(values, n_components):
    values = np.asarray(values, dtype=float)
    if values.ndim != 2 or values.shape[0] < 2 or values.shape[1] < 1:
        raise ValueError("t-SNE requires at least two vectors with one or more features.")
    if not np.isfinite(values).all():
        raise ValueError("t-SNE inputs must contain only finite numbers.")
    if n_components not in (2, 3):
        raise ValueError("Use 2 or 3 components for these visualizations.")

    return TSNE(
        n_components=n_components,
        perplexity=min(30, values.shape[0] - 1),
        init="random",
        learning_rate="auto",
        random_state=42,
    ).fit_transform(values)

In [ ]:
reduced_vectors_2d = project_vectors(vectors, n_components=2)
fig_2d = go.Figure(data=[go.Scatter(
    x=reduced_vectors_2d[:, 0],
    y=reduced_vectors_2d[:, 1],
    mode="markers",
    marker=dict(color=colors, opacity=0.8),
    text=hover_text,
    hoverinfo="text",
)])
# A 2D Scatter uses xaxis/yaxis; scene settings apply only to 3D traces.
fig_2d.update_layout(
    title="2D Chroma Chunk Embeddings (t-SNE)",
    xaxis_title="t-SNE 1",
    yaxis_title="t-SNE 2",
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40),
)
fig_2d.show()

In [ ]:
# Fit a separate projection rather than treating 2D coordinates as 3D data.
reduced_vectors_3d = project_vectors(vectors, n_components=3)
fig_3d = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors_3d[:, 0],
    y=reduced_vectors_3d[:, 1],
    z=reduced_vectors_3d[:, 2],
    mode="markers",
    marker=dict(size=5, color=colors, opacity=0.8),
    text=hover_text,
    hoverinfo="text",
)])
fig_3d.update_layout(
    title="3D Chroma Chunk Embeddings (t-SNE)",
    scene=dict(xaxis_title="t-SNE 1", yaxis_title="t-SNE 2", zaxis_title="t-SNE 3"),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40),
)
fig_3d.show()

**Part C summary:** each marker represents a chunk. Folder-derived colors and
hover excerpts make inspection easier. Projections are exploratory illustrations;
use labeled retrieval questions and source inspection to assess index quality.

## 11. Regression checks
Run after the pipeline. These assertions check source preservation, chunk token
limits, collection identity, stored-vector alignment, plot shapes and axes, and
small-corpus behavior. The two-vector test specifically covers the original
default-perplexity failure.

These are structural and smoke tests, not a benchmark of answer quality. The
384-dimension expectation is specific to the configured MiniLM model.

In [ ]:
assert len(documents) == len(files)
assert {Path(doc.metadata["source"]).resolve() for doc in documents} == {
    file.resolve() for file in files
}
source_text = {doc.metadata["source"]: doc.page_content for doc in documents}
for chunk in chunks:
    assert chunk.page_content in source_text[chunk.metadata["source"]]
    assert chunk.metadata["doc_type"] == Path(chunk.metadata["source"]).relative_to(knowledge_base_dir).parts[0]

assert max(chunk_token_counts) <= token_budget <= 256
assert set(vectorstore.get(include=[])["ids"]) == set(chunk_ids)
assert count == len(chunks) == len(chunk_ids)
assert vectors.shape == (count, 384)
assert np.allclose(np.linalg.norm(vectors, axis=1), 1.0, atol=1e-5)
assert len(colors) == len(hover_text) == count
assert set(doc_types) <= COLOR_MAP.keys()
assert reduced_vectors_2d.shape == (count, 2)
assert reduced_vectors_3d.shape == (count, 3)
assert np.isfinite(reduced_vectors_2d).all() and np.isfinite(reduced_vectors_3d).all()
assert fig_2d.layout.xaxis.title.text == "t-SNE 1"
assert fig_3d.layout.scene.zaxis.title.text == "t-SNE 3"
assert len(matches) == min(3, count)
assert all(match.page_content in {chunk.page_content for chunk in chunks} for match in matches)

# Two vectors must work for both plots, even with only two original features.
for output_dimensions in (2, 3):
    small_projection = project_vectors(np.array([[0.0, 1.0], [1.0, 0.0]]), output_dimensions)
    assert small_projection.shape == (2, output_dimensions)
    assert np.isfinite(small_projection).all()

for invalid_values in (np.ones((1, 4)), np.array([[np.nan], [0.0]])):
    try:
        project_vectors(invalid_values, 2)
    except ValueError:
        pass  # These inputs are deliberately invalid; success would be a regression.
    else:
        raise AssertionError("Invalid t-SNE input was accepted.")

print("Document, chunk, collection, retrieval, and visualization checks passed.")

Document, chunk, collection, retrieval, and visualization checks passed.


## Conclusion
You now have a local semantic-search index with inspectable source metadata:
1. Resolve stable paths and load categorized Markdown files.
2. Measure corpus size without confusing OpenAI-style and MiniLM token counts.
3. Split to a model-compatible token budget while preserving provenance.
4. Embed locally and persist a non-destructive, named Chroma snapshot.
5. Inspect retrieval results and visualize aligned vectors in 2D/3D.
6. Run assertions to catch common loading, chunking, storage, and plotting errors.

### Troubleshooting reference
| Symptom | Action |
|---|---|
| Tokenizer fork warning | Restart the kernel; run the first cell before any tokenizer use. Keep parallelism `false`. |
| Missing corpus or no files | Check your local `my-work/Week 5/knowledge-base` copy and category subfolders. |
| Model download fails | Check connectivity or populate the Hugging Face model cache; do not treat this as an API-key problem. |
| Different corpus snapshot | Choose a new collection name; this notebook intentionally does not delete old data. |
| Dimension mismatch | Use the same embedding model for indexing and querying, with a separate collection per configuration. |
| Small-corpus t-SNE failure | Use `project_vectors`; at least two vectors are required. |
| Plot does not display | Check the VS Code notebook renderer; try `fig_2d.show(renderer="browser")` locally. |

### Limitations and next steps
This loads and plots the entire small corpus in memory. Larger datasets need
batching, sampling, and index-lifecycle management. A failed initial insertion
may leave a partial collection; the next run rejects it rather than silently
assuming success. Choose a new collection name to retry without deleting data.
Normalized embeddings and attractive plots do not guarantee useful retrieval.
Future work includes a labeled retrieval evaluation set, source citations,
retrieval-augmented answer generation, context budgeting, and access controls.

**Reference rule:** keep loading, tokenization, embedding, storage, retrieval,
and visualization failures separate so each can be reproduced and tested.